In [1]:
# Import section
from backtesting import Backtest
import pandas as pd
from backtesting import Strategy
from backtesting.lib import crossover
import numpy as np
import math

# META
MARGIN = 0.01
CASH = 10_000
COMMISSION = 0.00002
table_name = 'f0_raw'

# DEFINE
signal_arr = []

/home/chris/backtester/dema_strat/.venv/lib/python3.14/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

# Strategy

In [2]:
# Define the EMA Strategy
class STRAT(Strategy):
    n1 = 9  # Fast EMA period
    n2 = 21  # Slow EMA period

    risk_pct = 0.01      # Risk 1% of account per trade
    risk_reward = 1      # TP = 2x the SL distance

    def init(self):
        super().init()

        # Calculate EMAs using Pandas directly
        close = pd.Series(self.data.Close)

        self.ema1 = self.I(lambda: close.ewm(span=self.n1, adjust=False).mean(), name="ema fast")
        self.ema2 = self.I(lambda: close.ewm(span=self.n2, adjust=False).mean(), name="ema slow")

    def next(self):
        price = self.data.Close[-1]

        prev_low  = self.data.Low[-2]
        prev_high = self.data.High[-2]

        long_signal  = crossover(self.ema1, self.ema2) 
        short_signal = crossover(self.ema2, self.ema1)

        risk_amt = max(self.equity, CASH) * self.risk_pct

        def get_size_by_leveraged(sl_dist):
            ideal_size = risk_amt / sl_dist
            # We subtract a buffer 85%
            max_leverage_size = (self.equity * 0.85) / (price * MARGIN)
            size = int(min(ideal_size, max_leverage_size))
            return size

        if long_signal:

            sl_dist = abs(price - prev_low)
            tp_dist = sl_dist * self.risk_reward

            if sl_dist == 0:
                signal_arr.append((self.data.index[-1], sl_dist, 0, 0))
                return

            size = get_size_by_leveraged(sl_dist)
            signal_arr.append((self.data.index[-1], sl_dist))

            if size <= 0:
                return

            self.position.close()
            self.buy(size=size, sl=price - sl_dist, tp=price + tp_dist)

        elif short_signal:

            sl_dist = abs(price - prev_high)
            tp_dist = sl_dist * self.risk_reward

            if sl_dist == 0:
                signal_arr.append((self.data.index[-1], sl_dist, 0, 0))
                return

            size = get_size_by_leveraged(sl_dist)
            signal_arr.append((self.data.index[-1], sl_dist))

            if size <= 0:
                return


            self.position.close()
            self.sell(size=size, sl=price + sl_dist, tp=price - tp_dist)


# Basic backtest

In [ ]:
# Load sample data and run a basic backtest
file_path = "~/market_data/XAUUSD/XAUUSD_M5.csv"

try:
  # Load data
  data = pd.read_csv(file_path, parse_dates=['Datetime'], index_col='Datetime')
  
  # Run backtest
  bt = Backtest(data, STRAT, cash=CASH, commission=COMMISSION, margin=MARGIN)
  stats = bt.run()
  
  print(stats)
  print("Profit factor: ", stats['Profit Factor'])
  
except Exception as e:
  print(f"Error: {str(e)}")

Start                     2020-01-02 01:00:00
End                       2025-12-31 23:55:00
Duration                   2190 days 22:55:00
Exposure Time [%]                     1.12248
Equity Final [$]                     17.81154
Equity Peak [$]                   10778.92692
Commissions [$]                    5384.93246
Return [%]                          -99.82188
Buy & Hold Return [%]               184.03997
Return (Ann.) [%]                   -64.61149
Volatility (Ann.) [%]                24.44087
CAGR [%]                            -65.19276
Sharpe Ratio                         -2.64358
Sortino Ratio                        -1.07556
Calmar Ratio                         -0.64718
Alpha [%]                          -103.59044
Beta                                  0.02048
Max. Drawdown [%]                   -99.83476
Avg. Drawdown [%]                    -5.97987
Max. Drawdown Duration     2178 days 08:35:00
Avg. Drawdown Duration      104 days 06:35:00
# Trades                          

# SQLite connection

In [5]:
import sqlite3
import os
from datetime import datetime

# Setup SQLite database
db_path = "../backtest_results.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print(f"Database setup complete at {db_path}")

conn.commit()

Database setup complete at ../backtest_results.db


# Combination test

In [7]:
from itertools import product
from collections import namedtuple


cursor.execute(f'''
CREATE TABLE IF NOT EXISTS {table_name} (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    asset TEXT NOT NULL,
    timeframe TEXT NOT NULL,
    n1 INTEGER NOT NULL,
    n2 INTEGER NOT NULL,
    risk_pct REAL NOT NULL,
    risk_reward REAL NOT NULL,
    return_pct REAL,
    max_drawdown_pct REAL,
    cagr_pct REAL,
    sharpe_ratio REAL,
    profit_factor REAL,
    win_rate_pct REAL,
    num_trades INTEGER,
    equity_final REAL,
    run_timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
)
''')

# Test all asset/timeframe combinations
assets = ['XAUUSD', 'EURUSD', 'GBPUSD', 'USDCAD', 'AUDUSD', 'USDCHF', 'US100', 'US500']
timeframes = ['M1', 'M5', 'M15', 'H1', 'H4', 'D1']

# Strategy parameters (from current STRAT class)
init_params = {
    'n1': 9,
    'n2': 21,
    'risk_pct': 0.01,
    'risk_reward': 1
}

parameters = [
    timeframes,
    assets,
    range(1, 6),  # rr
    [0.005, 0.01] # risk percent
]

total_combinations = math.prod(len(p) for p in parameters)
print(f"Testing {total_combinations} combinations...")

for timeframe, asset, rr, risk_pct in product(*parameters):
    file_path = f"~/market_data/{asset}/{asset}_{timeframe}.csv"
    
    try:
        # Load data
        data = pd.read_csv(file_path, parse_dates=['Datetime'], index_col='Datetime')
        
        # Run backtest
        bt = Backtest(data, STRAT, cash=CASH, commission=COMMISSION, margin=MARGIN)
        stats = bt.run(risk_reward=rr, risk_pct=risk_pct)
        
        # Extract metrics
        return_pct = stats['Return [%]']
        max_drawdown_pct = stats['Max. Drawdown [%]']
        cagr_pct = stats['CAGR [%]']
        sharpe_ratio = stats['Sharpe Ratio']
        profit_factor = stats['Profit Factor']
        win_rate_pct = stats['Win Rate [%]']
        num_trades = stats['# Trades']
        equity_final = stats['Equity Final [$]']
        
        # Store in database
        cursor.execute(f'''
            INSERT INTO {table_name} 
            (asset, timeframe, n1, n2, risk_pct, risk_reward, return_pct, max_drawdown_pct, cagr_pct, sharpe_ratio, profit_factor, win_rate_pct, num_trades, equity_final)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (asset, timeframe, init_params['n1'], init_params['n2'], risk_pct, rr, return_pct, max_drawdown_pct, cagr_pct, sharpe_ratio, profit_factor, win_rate_pct, num_trades, equity_final))
        
        conn.commit()
        
        print(f"✓ {asset}_{timeframe}: Return={return_pct:.2f}%, DD={max_drawdown_pct:.2f}%, Sharpe={sharpe_ratio:.2f}, WinRate={win_rate_pct:.2f}%")
        
    except Exception as e:
        print(f"✗ {asset}_{timeframe}: Error - {str(e)}")
        # Store error record
        cursor.execute(f'''
            INSERT INTO {table_name} 
            (asset, timeframe, n1, n2, risk_pct, risk_reward, return_pct, max_drawdown_pct, cagr_pct, sharpe_ratio, profit_factor, win_rate_pct, num_trades, equity_final)
            VALUES (?, ?, ?, ?, ?, ?, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL)
        ''', (asset, timeframe, init_params['n1'], init_params['n2'], risk_pct, rr))
        conn.commit()

print("\nBacktesting complete!")
print(f"Results stored in {db_path}")

Testing 480 combinations...
✓ XAUUSD_M1: Return=-99.86%, DD=-99.86%, Sharpe=-3.46, WinRate=42.86%
✓ XAUUSD_M1: Return=-99.85%, DD=-99.85%, Sharpe=-3.17, WinRate=43.17%
✓ XAUUSD_M1: Return=-99.88%, DD=-99.88%, Sharpe=-1.43, WinRate=32.20%
✓ XAUUSD_M1: Return=-99.88%, DD=-99.90%, Sharpe=-3.17, WinRate=31.63%
✓ XAUUSD_M1: Return=-99.84%, DD=-99.85%, Sharpe=-1.44, WinRate=27.07%
✓ XAUUSD_M1: Return=-99.84%, DD=-99.88%, Sharpe=-1.73, WinRate=26.98%
✓ XAUUSD_M1: Return=-99.86%, DD=-99.87%, Sharpe=-2.58, WinRate=25.09%
✓ XAUUSD_M1: Return=-99.86%, DD=-99.90%, Sharpe=-2.77, WinRate=24.72%
✓ XAUUSD_M1: Return=-99.84%, DD=-99.85%, Sharpe=-1.41, WinRate=24.01%
✓ XAUUSD_M1: Return=-99.86%, DD=-99.89%, Sharpe=-2.72, WinRate=23.99%
✓ EURUSD_M1: Return=-100.00%, DD=-100.00%, Sharpe=-8.76, WinRate=40.79%
✓ EURUSD_M1: Return=-100.00%, DD=-100.00%, Sharpe=-8.71, WinRate=40.80%
✓ EURUSD_M1: Return=-100.00%, DD=-100.00%, Sharpe=-8.64, WinRate=29.70%
✓ EURUSD_M1: Return=-100.00%, DD=-100.00%, Sharpe=-8.64,

/tmp/ipykernel_1358/3706652368.py:56: UserWarning: Some prices are larger than initial cash value. Note that fractional trading is not supported by this class. If you want to trade Bitcoin, increase initial cash, or trade μBTC or satoshis instead (see e.g. class `backtesting.lib.FractionalBacktest`.
  bt = Backtest(data, STRAT, cash=CASH, commission=COMMISSION, margin=MARGIN)


✓ US100_M1: Return=-98.84%, DD=-98.84%, Sharpe=-2.82, WinRate=45.02%


/tmp/ipykernel_1358/3706652368.py:56: UserWarning: Some prices are larger than initial cash value. Note that fractional trading is not supported by this class. If you want to trade Bitcoin, increase initial cash, or trade μBTC or satoshis instead (see e.g. class `backtesting.lib.FractionalBacktest`.
  bt = Backtest(data, STRAT, cash=CASH, commission=COMMISSION, margin=MARGIN)


✓ US100_M1: Return=-98.83%, DD=-98.83%, Sharpe=-2.02, WinRate=45.59%


...

✓ USDCHF_D1: Return=-10.88%, DD=-21.60%, Sharpe=-0.23, WinRate=18.18%
✓ US100_D1: Return=-0.51%, DD=-0.51%, Sharpe=-0.44, WinRate=0.00%
✓ US100_D1: Return=-1.96%, DD=-1.96%, Sharpe=-0.74, WinRate=0.00%
✓ US100_D1: Return=-0.51%, DD=-0.51%, Sharpe=-0.44, WinRate=0.00%
✓ US100_D1: Return=-1.96%, DD=-1.96%, Sharpe=-0.74, WinRate=0.00%
✓ US100_D1: Return=-0.51%, DD=-0.51%, Sharpe=-0.44, WinRate=0.00%
✓ US100_D1: Return=-1.96%, DD=-1.96%, Sharpe=-0.74, WinRate=0.00%


/tmp/ipykernel_1358/3706652368.py:56: UserWarning: Some prices are larger than initial cash value. Note that fractional trading is not supported by this class. If you want to trade Bitcoin, increase initial cash, or trade μBTC or satoshis instead (see e.g. class `backtesting.lib.FractionalBacktest`.
  bt = Backtest(data, STRAT, cash=CASH, commission=COMMISSION, margin=MARGIN)
/tmp/ipykernel_1358/3706652368.py:56: UserWarning: Some prices are larger than initial cash value. Note that fractional trading is not supported by this class. If you want to trade Bitcoin, increase initial cash, or trade μBTC or satoshis instead (see e.g. class `backtesting.lib.FractionalBacktest`.
  bt = Backtest(data, STRAT, cash=CASH, commission=COMMISSION, margin=MARGIN)
/tmp/ipykernel_1358/3706652368.py:56: UserWarning: Some prices are larger than initial cash value. Note that fractional trading is not supported by this class. If you want to trade Bitcoin, increase initial cash, or trade μBTC or satoshis ins

✓ US100_D1: Return=-0.51%, DD=-0.51%, Sharpe=-0.44, WinRate=0.00%
✓ US100_D1: Return=-1.96%, DD=-1.96%, Sharpe=-0.74, WinRate=0.00%
✓ US100_D1: Return=-0.51%, DD=-0.51%, Sharpe=-0.44, WinRate=0.00%
✓ US100_D1: Return=-1.96%, DD=-1.96%, Sharpe=-0.74, WinRate=0.00%
✓ US500_D1: Return=2.05%, DD=-0.28%, Sharpe=0.93, WinRate=100.00%
✓ US500_D1: Return=10.09%, DD=-1.89%, Sharpe=0.94, WinRate=71.43%
✓ US500_D1: Return=2.07%, DD=-0.67%, Sharpe=0.59, WinRate=60.00%


/tmp/ipykernel_1358/3706652368.py:56: UserWarning: Some prices are larger than initial cash value. Note that fractional trading is not supported by this class. If you want to trade Bitcoin, increase initial cash, or trade μBTC or satoshis instead (see e.g. class `backtesting.lib.FractionalBacktest`.
  bt = Backtest(data, STRAT, cash=CASH, commission=COMMISSION, margin=MARGIN)


✓ US500_D1: Return=4.06%, DD=-6.08%, Sharpe=0.25, WinRate=40.74%
✓ US500_D1: Return=1.53%, DD=-1.50%, Sharpe=0.33, WinRate=40.00%
✓ US500_D1: Return=3.30%, DD=-5.78%, Sharpe=0.17, WinRate=29.63%
✓ US500_D1: Return=2.46%, DD=-1.48%, Sharpe=0.49, WinRate=40.00%
✓ US500_D1: Return=7.54%, DD=-5.21%, Sharpe=0.35, WinRate=28.57%
✓ US500_D1: Return=3.39%, DD=-1.47%, Sharpe=0.59, WinRate=40.00%
✓ US500_D1: Return=16.70%, DD=-4.64%, Sharpe=0.69, WinRate=31.03%

Backtesting complete!
Results stored in ../backtest_results.db


### Summary

In [8]:
query_pf = f"""
SELECT asset, timeframe, risk_reward, return_pct, profit_factor, win_rate_pct, num_trades, equity_final
FROM {table_name}
WHERE profit_factor IS NOT NULL
ORDER BY profit_factor DESC
LIMIT 20
"""
top_pf = pd.read_sql_query(query_pf, conn)

query_wr = f"""
SELECT asset, timeframe, risk_reward, return_pct, profit_factor, win_rate_pct, num_trades, equity_final
FROM {table_name}
WHERE win_rate_pct IS NOT NULL
ORDER BY win_rate_pct DESC
LIMIT 20
"""
top_wr = pd.read_sql_query(query_wr, conn)

print("Top 20 by Profit Factor")
print(top_pf.to_string(index=False))
print(f"Average profit factor (top 20): {top_pf['profit_factor'].mean():.4f}")
print(f"Average win rate (top 20 by profit factor): {top_pf['win_rate_pct'].mean():.2f}%")
print("\nTop 20 by Win Rate")
print(top_wr.to_string(index=False))
print(f"Average profit factor (top 20 by win rate): {top_wr['profit_factor'].mean():.4f}")
print(f"Average win rate (top 20): {top_wr['win_rate_pct'].mean():.2f}%")


Top 20 by Profit Factor
 asset timeframe  risk_reward  return_pct  profit_factor  win_rate_pct  num_trades  equity_final
 US500        D1          5.0    3.392012       3.753201     40.000000           5  10339.201178
 US500        D1          2.0    2.071986       3.358676     60.000000           5  10207.198586
 US500        D1          4.0    2.459013       2.996924     40.000000           5  10245.901300
GBPUSD        D1          5.0   14.906615       2.334311     33.898305          59  11490.661492
GBPUSD        D1          5.0   31.096045       2.334311     33.898305          59  13109.604528
 US500        D1          3.0    1.526014       2.240647     40.000000           5  10152.601422
GBPUSD        D1          4.0   12.241564       2.151394     35.593220          59  11224.156354
GBPUSD        D1          4.0   25.341020       2.151394     35.593220          59  12534.101964
 US500        D1          1.0   10.092969       2.142519     71.428571          28  11009.296920
GBPUSD

In [ ]:
bt.plot(open_browser=False, filename=f"{table_name}.html", resample=False, plot_pl=False, plot_equity=False, plot_volume=False)